In [1]:
%load_ext autoreload
%autoreload 2

import sys
import pandas as pd
sys.path.insert(1, '../')

In [2]:
import pyFBS

from scipy import sparse
from xarray import DataArray
from scipy.sparse.linalg import eigsh
import numpy as np

import pyvista as pv
import pyansys

from pyFBS.utility import *


In [18]:
pyFBS.example_lab_testbench["meas"]

{'xlsx': '..\\data\\lab_testbench\\Measurements\\loc_file.xlsx',
 'Y_A': '..\\data\\lab_testbench\\Measurements\\Y_A.p',
 'Y_B': '..\\data\\lab_testbench\\Measurements\\Y_B.p',
 'Y_AB': '..\\data\\lab_testbench\\Measurements\\Y_AB.p'}

In [19]:
pyFBS.example_lab_testbench

{'STL': {'A': '..\\data\\lab_testbench\\STL\\A.stl',
  'B': '..\\data\\lab_testbench\\STL\\B.stl',
  'AB': '..\\data\\lab_testbench\\STL\\AB.stl'},
 'meas': {'xlsx': '..\\data\\lab_testbench\\Measurements\\loc_file.xlsx',
  'Y_A': '..\\data\\lab_testbench\\Measurements\\Y_A.p',
  'Y_B': '..\\data\\lab_testbench\\Measurements\\Y_B.p',
  'Y_AB': '..\\data\\lab_testbench\\Measurements\\Y_AB.p'},
 'FEM': {'A_rst': '..\\data\\lab_testbench\\FEM\\A\\file.rst',
  'A_full': '..\\data\\lab_testbench\\FEM\\A\\file.full',
  'B_rst': '..\\data\\lab_testbench\\FEM\\B\\file.rst',
  'B_full': '..\\data\\lab_testbench\\FEM\\B\\file.full',
  'AB_rst': '..\\data\\lab_testbench\\FEM\\AB\\file.rst',
  'AB_full': '..\\data\\lab_testbench\\FEM\\AB\\file.full'}}

In [20]:
pyFBS.example_auto_testbench

{'STL': {'ts': '..\\data\\automotive_testbench\\STL\\ts.stl',
  'transmission_mount': '..\\data\\automotive_testbench\\STL\\transmission_mount.stl',
  'shaker_only': '..\\data\\automotive_testbench\\STL\\shaker_only.stl',
  'roll_mount': '..\\data\\automotive_testbench\\STL\\roll_mount.stl',
  'receiver': '..\\data\\automotive_testbench\\STL\\receiver.stl',
  'engine_mount': '..\\data\\automotive_testbench\\STL\\engine_mount.stl'},
 'meas': {'xlsx_A': '..\\data\\automotive_testbench\\Measurements\\A.xlsx',
  'xlsx_AB_ref': '..\\data\\automotive_testbench\\Measurements\\AB_ref.xlsx',
  'xlsx_B_ref': '..\\data\\automotive_testbench\\Measurements\\B_ref.xlsx',
  'xlsx_BTS': '..\\data\\automotive_testbench\\Measurements\\BTS.xlsx',
  'xlsx_TS': '..\\data\\automotive_testbench\\Measurements\\TS.xlsx',
  'Y_A': '..\\data\\automotive_testbench\\Measurements\\A.p',
  'Y_AB_ref': '..\\data\\automotive_testbench\\Measurements\\AB_ref.p',
  'Y_B_ref': '..\\data\\automotive_testbench\\Measurements

In [6]:
stl = pyFBS.example_lab_testbench["STL"]["AB"]
xlsx = pyFBS.example_lab_testbench["meas"]["xlsx"]

full_file = pyFBS.example_lab_testbench["FEM"]["AB_full"]
ress_file = pyFBS.example_lab_testbench["FEM"]["AB_rst"]

In [7]:
MK = pyFBS.MK_model(ress_file,full_file,no_modes = 100,recalculate = False)

In [8]:
view3D = pyFBS.display.view3D(show_origin= True)

In [9]:
#view3D = pyFBS.display.view3D()
view3D.add_stl(stl,name = "engine_mount",color = "#8FB1CC",opacity = 0.1)

In [10]:
view3D.plot.add_mesh(MK.mesh, scalars = np.ones(MK.mesh.points.shape[0]),show_scalar_bar = False,name = "mesh",cmap = "coolwarm",show_edges = True,render_points_as_spheres  = True,style = "surface")

(vtkRenderingOpenGL2Python.vtkOpenGLActor)000001829FE811C8

In [11]:
select_mode = 2
_modeshape = MK.get_modeshape(select_mode)

mode_dict = dict_animation(_modeshape,"modeshape",pts = MK.pts,mesh = MK.mesh)
view3D.add_modeshape(mode_dict,run_animation = True)

In [12]:
df_imp = pd.read_excel(xlsx, sheet_name='Impacts_AB')
view3D.show_imp(df_imp,overwrite = True)
#view3D.label_imp(df_imp)
#df_imp

In [13]:
df_chn = pd.read_excel(xlsx, sheet_name='Channels_AB')
view3D.show_chn(df_chn)
#df_chn

In [14]:
df_chn_up = MK.update_locations_df(df_chn)
df_imp_up = MK.update_locations_df(df_imp)

view3D.show_imp(df_imp_up, color = "k",overwrite = False)


In [15]:
MK.FRF_synth(df_chn,df_imp,modal_damping = 0.003,frf_type = "accelerance")

In [ ]:
freq, Y_AB_exp = np.load(r"../data/lab_testbench/Measurements/Y_AB.p",allow_pickle = True)

In [ ]:
plt.figure(figsize = (12,8))

s1 = 5
s2 = 0

display(df_chn.iloc[[s1]])
display(df_imp.iloc[[s2]])

plt.subplot(211)
plt.semilogy(MK.freq,np.abs(MK.FRF[:,s1,s2]))
plt.semilogy(freq,np.abs(Y_AB_exp[s1,s2]))


plt.subplot(413)
plt.plot(MK.freq,np.angle(MK.FRF[:,s1,s2]))
plt.plot(freq,np.angle(Y_AB_exp[s1,s2]))



In [ ]:
#df_chn = pd.read_excel(xlsx, sheet_name='Channels_AB')
#df_imp = pd.read_excel(xlsx, sheet_name='Impacts_AB')

df_chn = df_chn_up
df_imp = df_imp_up

df_vp = pd.read_excel(xlsx, sheet_name='VP_Channels')
df_vpref = pd.read_excel(xlsx, sheet_name='VP_RefChannels')

vpt = pyFBS.VPT(df_chn,df_imp,df_vp,df_vpref)

In [ ]:
frf_exp = np.transpose(Y_AB_exp,(2,0,1))

vpt.apply_VPT(MK.FRF,MK.FRF)
vpt.consistency([1],[1])

In [ ]:
vpt.vptData.shape

In [ ]:
plt.semilogy(np.abs(vpt.u[8,:]))
plt.semilogy(np.abs(vpt.u_f[8,:]))

In [ ]:
plt.bar(range(9),vpt.specific_sensor)

In [ ]:
plt.bar(range(9),vpt.specific_impact)

In [ ]:
Y_SEMM_coh = np.zeros((24,24))

for i in range(24):
    for j in range(24):
        Y_SEMM_coh[i,j] = coh_frf(vpt.vptData[:,i,j], vpt.vptData[:,j,i])

plt.imshow(Y_SEMM_coh)